*0.1 Python for GenAI*

# retries with tenacity

**The situation.** One request in two hundred to the provider fails: a 429 "too busy", or a dropped connection. Each one becomes an error in a customer's chat window and a support ticket — although the very same request, sent one second later, would have worked. Meanwhile a *wrong API key* gets the same treatment as a busy signal: three slow retries before anyone is told the key is bad.

**The fix: retry automatically, but only for problems that can go away.** `tenacity` wraps a function with a policy: which errors to retry, how long to wait between tries, when to give up. Busy signals and network hiccups get retried with growing waits. A wrong key fails immediately.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The policy.** `is_transient` decides what is worth retrying. The decorator applies the waiting and the cap. Every retry writes a warning line — the trail you read when a provider has a bad hour.

In [2]:
import logging
import os
import time

import httpx
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception,
    stop_after_attempt,
    wait_random_exponential,
)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)
log = logging.getLogger("retry")


def is_transient(error: BaseException) -> bool:
    if isinstance(error, (httpx.TimeoutException, httpx.ConnectError)):
        return True  # network trouble: try again
    if isinstance(error, httpx.HTTPStatusError):
        return error.response.status_code in (
            429,
            500,
            502,
            503,
            504,
        )  # busy or broken server: try again
    return False  # anything else (401 wrong key, 400 bad request): give up now


attempts = []


@retry(
    retry=retry_if_exception(is_transient),
    wait=wait_random_exponential(multiplier=0.3, max=5),  # 0.3 s, 0.6 s, 1.2 s … plus a random bit
    stop=stop_after_attempt(4),
    before_sleep=before_sleep_log(log, logging.WARNING),
    reraise=True,
)
def ask_provider(timeout: float, api_key: str) -> str:
    attempts.append(time.perf_counter())
    with httpx.Client(timeout=timeout) as http:
        response = http.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={
                "model": MODEL,
                "messages": [{"role": "user", "content": "Reply with the single word: pong"}],
                "max_tokens": 3,
            },
        )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


print("healthy call:", ask_provider(30, os.environ["OPENAI_API_KEY"]), "| attempts:", len(attempts))
assert len(attempts) == 1

healthy call: Pong | attempts: 1


**Reading the output.** A healthy call took one attempt. Now the two failure cases — a network problem, and a wrong key.

In [3]:
attempts.clear()
try:
    ask_provider(
        0.001, os.environ["OPENAI_API_KEY"]
    )  # 1 ms: cannot succeed, looks like a network problem
except httpx.TimeoutException:
    print("network problem: attempts", len(attempts), "→ retried with growing waits, then gave up")
    transient_attempts = len(attempts)

attempts.clear()
try:
    ask_provider(30, "sk-invalid")
except httpx.HTTPStatusError as error:
    print("wrong key:", error.response.status_code, "→ attempts", len(attempts), "→ not retried")
    permanent_attempts = len(attempts)
assert transient_attempts == 4 and permanent_attempts == 1

WARNING Retrying __main__.ask_provider in 0.0303 seconds as it raised ConnectTimeout: timed out.


WARNING Retrying __main__.ask_provider in 0.167 seconds as it raised ConnectTimeout: timed out.


WARNING Retrying __main__.ask_provider in 1.15 seconds as it raised ConnectTimeout: timed out.


network problem: attempts 4 → retried with growing waits, then gave up


wrong key: 401 → attempts 1 → not retried


**Reading the output.** The network problem was retried four times (the warning lines show the growing waits) before giving up. The wrong key failed on the first attempt — no retries, no wasted seconds.

```
request ──▶ 429 / 5xx / timeout ──▶ wait 0.3 s → 0.6 s → 1.2 s (+ random) ──▶ request again … 4th failure → give up
request ──▶ 401 / 400 ──────────▶ fail now
request ──▶ 200 ────────────────▶ done
```

**The rule to remember.** Retry the weather, not the wrong address: busy signals and hiccups, never bad keys or bad requests.

| Use it when | Don't when | Instead use |
|---|---|---|
| calls to anything that fails briefly: providers, databases, queues | the error is permanent | the SDK's built-in `max_retries` for the simple case; a gateway retry policy for all services at once |

**Watch out**
- The random bit in the wait ("jitter") is not optional. Without it, 1,000 clients that failed together retry together and cause the next outage.
- The OpenAI SDK already retries twice. SDK × tenacity × a caller's loop = 27 attempts. Retry in one place.
- A retried POST can create two orders. Send an idempotency key so the server ignores the duplicate.